# Wine Quality multiclass classification

попробуем взять датасет Wine Quality, и разделить `quality` на 3 класса и обучить MLP

по классам:
- `0` — плохо (`quality <= 5`)
- `1` — среднее (`quality == 6`)
- `2` — хорошее (`quality >= 7`)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from minigrad.nn import Linear, ReLU, Sequential, BatchNormalization, Dropout
from minigrad.losses import CrossEntropyLoss
from minigrad.optim import Adam
from minigrad.data import DataLoader
from minigrad.trainer import Trainer
np.random.seed(42)


## 2. Загружаем датасет

`id=186` — Wine Quality из UCI Machine Learning Repository.

In [ ]:
wine = fetch_ucirepo(id=186)

X_df = wine.data.features.copy()
y_df = wine.data.targets.copy()

print("X shape:", X_df.shape)
print("y shape:", y_df.shape)


### Первые строки признаков

In [ ]:
X_df.head(10)


### Целевая переменная `quality`

In [ ]:
y_df.head(10)


### Базовая статистика

In [ ]:
X_df.describe().T


## 3. Превращаем `quality` в 3 класса

\[
class =
\begin{cases}
0, & quality \le 5 \\
1, & quality = 6 \\
2, & quality \ge 7
\end{cases}
\]


In [ ]:
quality = y_df.to_numpy().ravel()

y = np.zeros_like(quality, dtype=np.int64)
y[quality == 6] = 1
y[quality >= 7] = 2

class_names = np.array(["bad", "medium", "good"])

classes, counts = np.unique(y, return_counts=True)

class_table = pd.DataFrame({
    "class_id": classes,
    "class_name": class_names[classes],
    "count": counts,
    "share": counts / counts.sum(),
})

class_table


In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(class_table["class_name"], class_table["count"])
plt.xlabel("Class")
plt.ylabel("Samples")
plt.title("Wine quality class distribution")
plt.show()


## 4. Train/test split

In [ ]:
X = X_df.to_numpy(dtype=np.float64)

indices = np.arange(len(X))
np.random.shuffle(indices)

split = int(0.8 * len(X))
train_idx = indices[:split]
test_idx = indices[split:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("train:", X_train.shape, y_train.shape)
print("test :", X_test.shape, y_test.shape)


### Нормализация

\[
x' = \frac{x - \mu}{\sigma}
\]

Статистики считаем только по train.


In [ ]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1.0

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

pd.DataFrame({
    "feature": X_df.columns,
    "train_mean_after_norm": X_train.mean(axis=0),
    "train_std_after_norm": X_train.std(axis=0),
}).round(3)


## 5. DataLoader и модель

In [ ]:
train_loader = DataLoader(
    X_train,
    y_train,
    batch_size=64,
    shuffle=True,
)

model = Sequential(
    Linear(X_train.shape[1], 64),
    BatchNormalization(64),
    ReLU(),
    Dropout(0.2),
    Linear(64, 32),
    ReLU(),
    Linear(32, 3),
)

loss_fn = CrossEntropyLoss()
optimizer = Adam(model, lr=0.001)
trainer = Trainer(model, loss_fn, optimizer)

model


## 6. Обучение

In [ ]:
history = trainer.fit(train_loader, epochs=150)

print("loss:", history["loss"][0], "->", history["loss"][-1])


### График loss

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history["loss"])
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training loss")
plt.grid(True, alpha=0.3)
plt.show()


## 7. Проверяем на test

In [ ]:
model.eval()

logits = model(X_test)
pred = np.argmax(logits, axis=1)

accuracy = np.mean(pred == y_test)

print("test accuracy:", accuracy)


### Таблица первых предсказаний

In [ ]:
pred_table = pd.DataFrame({
    "true_id": y_test[:25],
    "true": class_names[y_test[:25]],
    "pred_id": pred[:25],
    "pred": class_names[pred[:25]],
})

pred_table


### Confusion matrix

In [ ]:
cm = np.zeros((3, 3), dtype=int)

for true_class, pred_class in zip(y_test, pred):
    cm[true_class, pred_class] += 1

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{name}" for name in class_names],
    columns=[f"pred_{name}" for name in class_names],
)

cm_df


In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.xticks(range(3), class_names)
plt.yticks(range(3), class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix")

for i in range(3):
    for j in range(3):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.show()


## 8. Итог

В примере использованы:
- `Linear`
- `BatchNormalization`
- `ReLU`
- `Dropout`
- `Sequential`
- `CrossEntropyLoss`
- `Adam`
- `DataLoader`
- `Trainer`
